# Aprendizaje por Refuerzo en StarCraft: Brood War
## Implementación con TorchCraft + PPO

**Autor:** Eduard Pérez  
**Fecha:** Mayo 2026  
**Estado:** Entrenamiento activo

> **Nota:** Este notebook es documentación de referencia del proyecto. El agente se ejecuta
> en un host Windows 10 con una VM VirtualBox Windows 7 que corre StarCraft BW.
> No puede ejecutarse directamente en Google Colab (requiere la VM y BWEnv.dll),
> pero todas las celdas de código son ilustrativas y sirven de referencia.

---

## Resumen

Este trabajo presenta el diseño, implementación y puesta en marcha de un agente de aprendizaje
por refuerzo (RL) capaz de jugar a *StarCraft: Brood War 1.16.1* en un escenario de
micro-management de combate. El agente se ejecuta en un host Windows 10 y se comunica
con el juego a través del protocolo TorchCraft ZMQ, que expone el estado estructurado
del juego (posiciones, HP, tipos de unidad) mediante BWAPI 4.4.0 inyectado en el proceso
de StarCraft dentro de una máquina virtual Windows 7.

El algoritmo de aprendizaje es **Proximal Policy Optimization (PPO)** con una red MLP
que consume un vector de observación de 189 características.
El escenario actual es `dragoons_zealots.scm` — 8 unidades Protoss vs. 8 unidades Protoss —,
un espejo simétrico donde BWEnv reporta `player_id=0` para todas las unidades,
lo que requiere identificación de equipos por posición.

---

## Índice

1. [Introducción](#introduccion)
2. [Entorno de Ejecución](#entorno)
3. [Pila de Software en la VM](#pila)
4. [Arquitectura del Sistema](#arquitectura)
5. [Cómo se Comunica el Modelo con BWAPI](#comunicacion)
6. [Protocolo TorchCraft](#protocolo)
7. [Espacio de Observación](#observacion)
8. [Espacio de Acciones](#acciones)
9. [Algoritmo PPO](#ppo)
10. [Recompensa](#recompensa)
11. [Errores y Soluciones (16 errores)](#errores)
12. [Uso Rápido](#uso)
13. [Recursos y Trabajo Futuro](#recursos)


## 1. Introducción <a id='introduccion'></a>

*StarCraft: Brood War* es uno de los entornos de referencia más complejos para agentes
artificiales. Su espacio de acciones masivo, la información imperfecta y la necesidad
de micro-gestión táctica en tiempo real lo convierten en un banco de pruebas ideal
para algoritmos de RL modernos.

Este proyecto se centra en el **micro-management de combate**: controlar 8 unidades Protoss
para derrotar a 8 unidades enemigas en el menor número de frames posible.

### ¿Por qué TorchCraft y no píxeles?

Un enfoque basado en captura de pantalla + OCR fue considerado inicialmente. Se descartó por:

| Problema | Detalle |
|---|---|
| **Latencia** | `VBoxManage screenshotpng` introduce ~150 ms por frame |
| **Fragilidad** | OCR dependiente de resolución, fuentes y región de captura |
| **Riqueza** | TorchCraft provee posiciones, HP exacto, tipo y flags sin ruido visual |

### Escenarios usados

| Fase | Mapa | Descripción |
|---|---|---|
| Integración | `m5v5_c_far.scm` | 5 Marines Terran vs. 5 Zerglings |
| Actual | `dragoons_zealots.scm` | 8 Protoss vs. 8 Protoss (espejo simétrico) |


## 2. Entorno de Ejecución <a id='entorno'></a>

### 2.1 Host

| Parámetro | Valor |
|---|---|
| Sistema operativo | Windows 10 Pro 22H2 |
| Python | 3.13 |
| CUDA | Disponible (entrenamiento en GPU) |
| Hypervisor | Oracle VirtualBox 7.x |
| Gestión VM | Vagrant |

### 2.2 Máquina Virtual

| Parámetro | Valor |
|---|---|
| SO invitado | Windows 7 Ultimate SP1 x64 |
| RAM asignada | 8 GB |
| CPUs asignadas | 4 |
| NIC 1 | NAT (10.0.2.15) — acceso a internet |
| NIC 2 | Bridge (192.168.0.23) — comunicación con host |

La VM se gestiona con Vagrant. El host se conecta a BWEnv a través del **adaptador bridge**
(`192.168.0.23:11111`), no mediante port-forwarding NAT (ver Error 7).


## 3. Pila de Software en la VM <a id='pila'></a>

```
StarCraft.exe  <--- ChaosLauncher (inyector)
     |
     +-- BWAPI 4.4.0 (hook dentro del proceso)
               |
               +-- BWEnv.dll (plugin de IA de BWAPI)
                         |
                         +-- ZMQ REP socket :11111
```

| Componente | Rol |
|---|---|
| StarCraft BW 1.16.1 | El juego base, corre en ventana 400×300 px |
| ChaosLauncher | Inyector de DLLs — carga BWAPI en el proceso de StarCraft |
| BWAPI 4.4.0 | Hook en proceso — expone la API del juego a plugins |
| BWEnv.dll | Plugin BWAPI — servidor ZMQ REP que serializa estado con FlatBuffers |

### bwapi.ini (configuración final)


In [ ]:
# bwapi.ini — guardar en: C:\Archivos de programa\Starcraft\bwapi-data\bwapi.ini
bwapi_ini = """
[ai]
ai     = bwapi-data/AI/BWEnv.dll
ai_dbg = bwapi-data/AI/BWEnv.dll

[auto_menu]
auto_menu    = SINGLE_PLAYER
auto_restart = ON
map          = Maps/BroodWar/micro/dragoons_zealots.scm
race         = Protoss
enemy_count  = 1
enemy_race   = Protoss
game_type    = USE_MAP_SETTINGS

[config]
shared_memory = ON

[window]
windowed = ON
width    = 400
height   = 300

[starcraft]
sound = OFF
"""
print("Contenido de bwapi.ini:")
print(bwapi_ini)


**Nota clave:** `game_type = USE_MAP_SETTINGS` es crítico — los mapas de micro-management
no tienen *starting locations* para modo MELEE, por lo que la partida se congela
indefinidamente con `game_type = MELEE` (ver Error 3).

### torchcraft.ini (configuración BWEnv.dll)


In [ ]:
# torchcraft.ini — guardar en: C:\Archivos de programa\Starcraft\bwapi-data\torchcraft.ini
torchcraft_ini = """
[general]
port = 11111
log_path = C:/tc_data/torchcraft_log_cpp_port_
display_log = false
img_mode = raw
window_mode = windows

[starcraft]
assume_on = true
launcher = bwheadless
"""
print("Contenido de torchcraft.ini:")
print(torchcraft_ini)
print()
print("CLAVE: assume_on = true indica a BWEnv que StarCraft ya esta corriendo")
print("(lanzado por ChaosLauncher). Con false, BWEnv intenta lanzarlo de nuevo -> deadlock)")


## 4. Arquitectura del Sistema <a id='arquitectura'></a>

```
Host Windows 10
|
+-- python main.py
|     |
|     +-- TorchCraftClient --ZMQ REQ--> 192.168.0.23:11111
|     |     +-- proto.py (FlatBuffers puro Python)
|     |
|     +-- SC1EnvTC (gymnasium.Env)
|     |     +-- StateEncoder  -> vector float32 (189 features)
|     |     +-- CommandExecutor -> lista de comandos BWAPI
|     |     +-- TCRewardCalculator -> recompensa por combate
|     |     +-- ActionLogger  -> logs/action_log.jsonl
|     |
|     +-- PPOAgent
|     |     +-- ActorCriticMLP  (189 -> 256 -> 256 -> actor/critic)
|     |     +-- RolloutBuffer   (GAE lambda=0.95)
|     |
|     +-- Trainer
|           +-- Rollout collection (2048 pasos)
|           +-- PPO update (4 epocas, mini-batch 64)
|           +-- Checkpoint cada 1000 pasos
|
+--------- ZMQ bridge 192.168.0.23:11111 --------->
                  |
+------------------v-------------------------------------------+
|          VM Windows 7 -- StarCraft BW 1.16.1                 |
|                                                              |
|  ChaosLauncher                                               |
|    +-- StarCraft.exe (ventana 400x300)                       |
|          +-- BWAPI 4.4.0 (hook en proceso)                   |
|                +-- BWEnv.dll                                 |
|                      +-- ZMQ REP :11111                      |
|                                                              |
|  Mapa: Maps/BroodWar/micro/dragoons_zealots.scm              |
|  8 Protoss (jugador 0)  vs.  8 Protoss (IA)                  |
+--------------------------------------------------------------+
```


## 5. Cómo se Comunica el Modelo con BWAPI <a id='comunicacion'></a>

### Flujo de decisión (red → juego)

```
ActorCriticMLP
|  recibe obs[189]
|  produce accion discreta in {0..64}
v
decode_tc_action(accion)          # action_space.py
|  0        -> TCAction(NOOP)
|  1..64    -> TCAction(ATTACK_MOVE, grid_row, grid_col)
v
CommandExecutor.build_commands(action, state)   # command_executor.py
|  Convierte grid (row,col) a pixeles (x,y):
|    x = (col + 0.5) / 8 * map_w_px
|    y = (row + 0.5) / 8 * map_h_px
|  Filtra unidades del equipo propio (_own_unit_ids)
|  Para cada unidad propia genera:
|    [21, uid, 1, -1, x, y, 0]
|     ^    ^   ^   ^  ^  ^  +-- extra (0)
|     |    |   |   |  +--+------ coordenadas en pixeles
|     |    |   |   +------------ target_uid (-1 = posicion, no unidad)
|     |    |   +---------------- BWAPI UnitCommandType::Attack_Move = 1
|     |    +-------------------- ID de unidad BWAPI
|     +------------------------- TC_CMD_UNIT_PROTECTED = 21
v
TorchCraftClient.send(commands)   # client.py
|  Serializa con FlatBuffers -> ZMQ REQ -> 192.168.0.23:11111
v
BWEnv.dll (VM)
|  Por cada comando code=21: BWAPI::Broodwar->getUnit(uid)->attack(pos)
v
StarCraft.exe
   Mueve la unidad y auto-ataca enemigos en rango
```

### Flujo de observación (juego → red)

```
StarCraft.exe -> avanza 1 frame (~42 ms a 24 FPS)
v
BWAPI 4.4.0 -> lee posiciones, HP, flags, recursos
v
BWEnv.dll -> serializa Frame/FrameDiff en FlatBuffers StateUpdate -> ZMQ REP
v
TorchCraftClient._process_state() -> GameState
v
SC1EnvTC.step()/reset() -> limpia unidades stale
v
StateEncoder.encode(state) -> obs[189]
v
TCRewardCalculator.compute(state, action) -> reward
v
ActorCriticMLP -> siguiente accion
```


In [ ]:
# Identificacion de equipos: split posicional (x+y diagonal)
# BWEnv reporta player_id=0 para TODAS las unidades (ver Error 12).
# Solucion: ordenar por x+y y dividir por la mediana.

def init_teams(state):
    """Divide unidades en equipo propio / enemigo por posicion inicial."""
    all_u = [
        u for units in state.units.values() for u in units.values()
        # excluir edificios, workers y recursos
    ]
    all_u.sort(key=lambda u: u.x + u.y)   # diagonal x+y
    mid = len(all_u) // 2
    own_unit_ids   = frozenset(u.id for u in all_u[:mid])   # esquina inf-izq
    enemy_unit_ids = frozenset(u.id for u in all_u[mid:])   # esquina sup-der
    return own_unit_ids, enemy_unit_ids

# Esta logica se implementa de forma IDENTICA en:
#   - TCRewardCalculator (sc1_rl/torchcraft/reward.py)
#   - StateEncoder       (sc1_rl/torchcraft/state_encoder.py)
#   - CommandExecutor    (sc1_rl/torchcraft/command_executor.py)
# y se resetea al inicio de cada episodio.


In [ ]:
# Formato de comando TorchCraft vs BWAPI
# ERROR CRITICO: confundir el codigo TorchCraft (game-level) con BWAPI UnitCommandType

# Codigos TorchCraft (Command.code):
TC_CODE_NOOP              = 0   # no-op
TC_CODE_QUIT              = 1   # QUIT -- termina la partida! NO enviar!
TC_CMD_UNIT_PROTECTED     = 21  # ejecutar comando BWAPI en unidad propia

# BWAPI UnitCommandType (dentro de args[1] cuando code=21):
BWAPI_ATTACK_MOVE         = 1
BWAPI_BUILD               = 2
BWAPI_TRAIN               = 4
BWAPI_MOVE                = 10
BWAPI_GATHER              = 15
BWAPI_RIGHT_CLICK_POS     = 30

# Formato correcto de ATTACK_MOVE:
#   [TC_CMD_UNIT_PROTECTED, uid, BWAPI_ATTACK_MOVE, target_uid, x, y, extra]
#   [21,                    uid, 1,                 -1,         x, y, 0    ]

def make_attack_move_cmd(uid, x, y):
    return [TC_CMD_UNIT_PROTECTED, uid, BWAPI_ATTACK_MOVE, -1, x, y, 0]

# Con code=1 (incorrecto): BWEnv.dll lo interpreta como QUIT -> partida termina
# Con code=21, args[1]=1 (correcto): BWEnv ejecuta attack_move en la unidad


## 6. Protocolo TorchCraft <a id='protocolo'></a>

El paquete Python `torchcraft` fue eliminado de PyPI. Este proyecto incluye una
**reimplementación completa en Python puro** usando `pyzmq` + `flatbuffers`.

### Estructura de mensaje FlatBuffers

```
Message {
  msg:  Any   (union discriminant en VT4, tabla en VT6)
  uid:  string (VT8)
}
```

**El orden de campos es CRITICO** (ver Error 6). Una inversión hace que BWEnv.dll
lea `msg_type` erróneo y descarte silenciosamente todos los mensajes.

### Tipos de mensaje

| Valor | Nombre | Dirección | Descripción |
|---|---|---|---|
| 1 | `HandshakeClient` | → servidor | Versión de protocolo (30) |
| 2 | `Commands` | → servidor | Lista de comandos BWAPI |
| 3 | `HandshakeServer` | ← servidor | Mapa, player_id, dimensiones |
| 4 | `StateUpdate` | ← servidor | Frame o FrameDiff con unidades |
| 5 | `PlayerLeft` | ← servidor | Jugador abandonó la partida |
| 6 | `EndGame` | ← servidor | Partida terminada |
| 7 | `Error` | ← servidor | Error en el servidor |

### Protocolo de handshake

```
Cliente (Python)                    BWEnv.dll (VM)
     |                                    |
     |-- HandshakeClient{protocol=30} --> |
     |<-- HandshakeServer{map, player_id, |
     |         map_w, map_h, lag} --------|  <- bloquea hasta recibir esto
     |                                    |
     |-- Commands{[...]} -------------->  |  (bucle de juego)
     |<-- StateUpdate{Frame|FrameDiff} ---|  <- un frame de juego
     |           ...                      |
     |-- Commands{[]}  (NOOP) ----------> |
     |<-- EndGame (msg_type=6) -----------|  <- partida terminada
```


In [ ]:
# Error 6 fix: orden correcto de campos FlatBuffers en _message()
# proto.py -> funcion _message()

# INCORRECTO (uid en slot 0 = VT4, msg en slots 1-2):
def _message_WRONG(b, uid_off, msg_type, inner_off):
    b.StartObject(3)
    b.PrependUOffsetTRelativeSlot(0, uid_off, 0)    # slot 0 -> VT4: uid  (INCORRECTO)
    b.PrependUint8Slot(1, msg_type, 0)              # slot 1 -> VT6: msg_type (INCORRECTO)
    b.PrependUOffsetTRelativeSlot(2, inner_off, 0)  # slot 2 -> VT8: msg
    return b.EndObject()

# CORRECTO (msg_type en slot 0 = VT4, msg en slot 1 = VT6, uid en slot 2 = VT8):
def _message_CORRECT(b, uid_off, msg_type, inner_off):
    b.StartObject(3)
    b.PrependUint8Slot(0, msg_type, 0)              # slot 0 -> VT4: msg_type discriminant
    b.PrependUOffsetTRelativeSlot(1, inner_off, 0)  # slot 1 -> VT6: msg table
    b.PrependUOffsetTRelativeSlot(2, uid_off, 0)    # slot 2 -> VT8: uid string
    return b.EndObject()

# Con el orden incorrecto, BWEnv leia el primer byte del string 'uid' como msg_type,
# obtenia un valor invalido y descartaba el HandshakeClient silenciosamente.
# Resultado: cliente esperaba respuesta indefinidamente (timeout 120s).


## 7. Espacio de Observación <a id='observacion'></a>

El `StateEncoder` convierte cada `GameState` en un vector `float32` de **189 características**:

| Rango | Categoría | Features | Descripción |
|---|---|---|---|
| [0:4] | Recursos | 4 | minerals, gas, supply_used, supply_max |
| [4:44] | Workers propios | 8 × 5 = 40 | x, y, hp_norm, is_idle, is_gathering |
| [44:104] | Armada propia | 12 × 5 = 60 | x, y, hp_norm, type_norm, is_attacking |
| [104:144] | Edificios propios | 8 × 5 = 40 | x, y, hp_norm, type_norm, is_training |
| [144:184] | Enemigos visibles | 10 × 4 = 40 | x, y, hp_norm, type_norm |
| [184:189] | Resumen global | 5 | n_workers, n_army, n_buildings, n_enemies, frame_norm |

Las coordenadas se normalizan dividiendo entre el tamaño del mapa en píxeles
(`map_w × 8`, `map_h × 8` — el mapa se mide en walk tiles de 8 px).

## 8. Espacio de Acciones <a id='acciones'></a>

Espacio discreto con **65 acciones** (combate puro, sin economía):

| Rango | Tipo | Cantidad | Descripción |
|---|---|---|---|
| 0 | NOOP | 1 | Sin acción — avanza un frame |
| 1–64 | ATTACK_MOVE | 64 | Ataque en grid 8×8 sobre el mapa completo |

El grid 8×8 divide el mapa en 64 celdas. La celda `(row, col)` se convierte a píxeles
tomando el centroide de la celda.


In [ ]:
# Conversion grid (row, col) -> coordenadas en pixeles
GRID_SIZE = 8  # cuadricula 8x8

def grid_to_pixels(grid_row, grid_col, map_w_walk_tiles, map_h_walk_tiles):
    """Convierte celda (row, col) del grid 8x8 a coordenadas en pixeles.
    
    map_w/h_walk_tiles: ancho/alto del mapa en WALK TILES (1 tile = 8 px).
    Los estados de TorchCraft reportan map_size en walk tiles.
    """
    map_w_px = int(map_w_walk_tiles) * 8
    map_h_px = int(map_h_walk_tiles) * 8
    x = int((grid_col + 0.5) / GRID_SIZE * map_w_px)
    y = int((grid_row + 0.5) / GRID_SIZE * map_h_px)
    return x, y

# Ejemplo: mapa 512x512 walk tiles (4096x4096 px)
# Celda (0,0) -> centroide (256, 256)
# Celda (7,7) -> centroide (3840, 3840)
for row, col in [(0,0), (0,7), (4,4), (7,7)]:
    x, y = grid_to_pixels(row, col, 512, 512)
    print(f"Grid ({row},{col}) -> Pixel ({x},{y})")


## 9. Algoritmo de Aprendizaje — PPO <a id='ppo'></a>

### 9.1 Red Neuronal (ActorCriticMLP)

```
Input: float32[189]
  |
  +-- Linear(189 -> 256) + LayerNorm + ReLU
  +-- Linear(256 -> 256) + LayerNorm + ReLU
  |
  +-- Actor:  Linear(256 -> 65) -> logits -> Categorical -> accion discreta
  +-- Critic: Linear(256 -> 1)  -> V(s)
```

Se usa `LayerNorm` en lugar de `BatchNorm` para estabilidad con batch_size=1
durante la recolección de rollouts.
Inicialización ortogonal: ganancia `sqrt(2)` para capas ocultas,
`0.01` para el actor (entropía alta = exploración uniforme), `1.0` para el crítico.


In [ ]:
import torch
import torch.nn as nn
import numpy as np

class ActorCriticMLP(nn.Module):
    """Red neuronal actor-critic para PPO en StarCraft BW."""

    OBS_SIZE = 189
    N_ACTIONS = 65  # 1 NOOP + 64 ATTACK_MOVE (grid 8x8)

    def __init__(self, hidden=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(self.OBS_SIZE, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),        nn.LayerNorm(hidden), nn.ReLU(),
        )
        self.actor  = nn.Linear(hidden, self.N_ACTIONS)
        self.critic = nn.Linear(hidden, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.shared.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, obs):
        x = self.shared(obs)
        return self.actor(x), self.critic(x).squeeze(-1)

    def get_action(self, obs, action=None):
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value

# Verificar dimensiones
net = ActorCriticMLP()
dummy_obs = torch.zeros(1, ActorCriticMLP.OBS_SIZE)
action, log_prob, entropy, value = net.get_action(dummy_obs)
print(f"Action: {action.item()}, Log prob: {log_prob.item():.4f}, Value: {value.item():.4f}")
print(f"Parametros totales: {sum(p.numel() for p in net.parameters()):,}")


In [ ]:
# PPO: bucle de entrenamiento simplificado (pseudocodigo ejecutable)

# Hiperparametros
ROLLOUT_STEPS  = 2048
MINI_BATCH     = 64
PPO_EPOCHS     = 4
GAMMA          = 0.99
GAE_LAMBDA     = 0.95
CLIP_EPS       = 0.2
LR             = 3e-4
COEF_VALUE     = 0.5
COEF_ENTROPY   = 0.01
MAX_GRAD_NORM  = 0.5

def compute_gae(rewards, values, last_value, dones, gamma=GAMMA, lam=GAE_LAMBDA):
    """Generalized Advantage Estimation."""
    advantages = []
    gae = 0.0
    for t in reversed(range(len(rewards))):
        next_val = last_value if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    return advantages

def ppo_loss(new_log_probs, old_log_probs, advantages, values_pred, returns, entropy):
    """Loss PPO: L_clip - coef_value * L_vf + coef_entropy * H."""
    ratio = (new_log_probs - old_log_probs).exp()
    surr1 = ratio * advantages
    surr2 = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * advantages
    L_clip = -torch.min(surr1, surr2).mean()
    L_vf   = ((values_pred - returns) ** 2).mean()
    H      = entropy.mean()
    return L_clip + COEF_VALUE * L_vf - COEF_ENTROPY * H

print(f"Hiperparametros PPO:")
print(f"  rollout_steps={ROLLOUT_STEPS}, mini_batch={MINI_BATCH}, epochs={PPO_EPOCHS}")
print(f"  gamma={GAMMA}, lambda={GAE_LAMBDA}, clip_eps={CLIP_EPS}")
print(f"  lr={LR}, coef_value={COEF_VALUE}, coef_entropy={COEF_ENTROPY}")


## 10. Recompensa <a id='recompensa'></a>

`TCRewardCalculator` calcula la recompensa por frame a partir del `GameState`:

| Señal | Valor | Descripción |
|---|---|---|
| Supervivencia | +0.001 / frame | Incentivo por mantenerse vivo |
| ATTACK_MOVE | +0.005 / frame | Incentivo por enviar comandos activos |
| NOOP | −0.003 / frame | Penalización por inacción |
| Daño infligido | +0.001 × HP_perdido | Daño causado al enemigo en el frame |
| Baja enemiga | +0.50 por baja | Unidad enemiga eliminada |
| Baja propia | −0.50 por baja | Unidad propia eliminada |
| Acercamiento | +0.0005 × Δdist | Mejora en distancia media al enemigo |
| Victoria | +10.0 | Todas las unidades enemigas eliminadas |
| Derrota | −5.0 | Todas las unidades propias eliminadas |

**Recompensa esperada con política aleatoria:** ≈ 12.0 por episodio de 2000 pasos
= 2000 × (attack_reward + survival) = 2000 × 0.006, sin kills ni muertes.

La función empieza a diferenciarse cuando el agente aprende a dirigir unidades
hacia el enemigo (señal de distancia y bajas).


## 11. Errores de Desarrollo y Soluciones <a id='errores'></a>

Esta sección documenta cronológicamente los 16 problemas encontrados durante la implementación.

---

### Error 1: DirectDraw 256 colores no disponible en VirtualBox
**Síntoma:** StarCraft arranca con pantalla en negro o colores distorsionados.  
**Causa:** VirtualBox VBoxVGA no emula DirectDraw de 256 colores.  
**Solución:** Instalar `cnc-ddraw` (wrapper DirectDraw) en la carpeta de StarCraft.

---

### Error 2: bwheadless incompatible con modo SINGLE_PLAYER
**Síntoma:** `auto_menu` llega al menú principal y se detiene, no puede crear partida.  
**Causa:** `bwheadless.exe` inicia StarCraft en modo LAN; `auto_menu = SINGLE_PLAYER` no funciona en modo LAN.  
**Solución:** Usar **ChaosLauncher** con plugin `BWAPI Injector 4.4.0 [RELEASE]`.

---

### Error 3: `game_type = MELEE` bloquea la partida en carga
**Síntoma:** La partida empieza a cargar pero se congela indefinidamente.  
**Causa:** Los mapas de micro (`dragoons_zealots.scm`) no tienen *starting locations* para MELEE.  
**Solución:** `game_type = USE_MAP_SETTINGS` en `bwapi.ini`.

---

### Error 4: `assume_on = false` congela StarCraft
**Síntoma:** StarCraft se congela en los primeros segundos tras el arranque.  
**Causa:** BWEnv intenta lanzar *otro* proceso StarCraft desde dentro del proceso ya en ejecución → deadlock.  
**Solución:** `assume_on = true` en `torchcraft.ini`.

---

### Error 5: `torchcraft.ini` en ubicación incorrecta
**Síntoma:** BWEnv no respeta las configuraciones aunque el archivo exista en `C:\tc_data\`.  
**Causa:** BWEnv busca el archivo en `C:\Archivos de programa\Starcraft\bwapi-data\torchcraft.ini`.  
**Solución:** Copiar el archivo a la ruta correcta relativa al directorio de StarCraft.


### Error 6: Orden FlatBuffers incorrecto — BWEnv nunca responde
**Síntoma:** Cliente envía HandshakeClient y espera 15–120 s sin respuesta.  
**Causa:** Campos `msg` y `uid` en orden incorrecto en VTable → BWEnv recibe `msg_type` inválido y descarta silenciosamente el mensaje.  
**Solución:** (ver celda de código anterior)

---

### Error 7: Firewall de Windows bloqueando el puerto 11111 + VirtualBox NAT
**Síntoma:** Después de corregir FlatBuffers, la conexión sigue sin establecerse.  
**Causa:** Firewall bloqueaba el puerto. Y port-forwarding NAT de VirtualBox no funciona bien para conexiones host→VM.  
**Solución:** Conectar directamente a la IP del bridge (`192.168.0.23:11111`), no via NAT (`127.0.0.1:11111`).

```yaml
# config.yaml
torchcraft:
  host: "192.168.0.23"  # Bridge adapter IP — bypasses VirtualBox NAT
  port: 11111
```

---

### Error 8: Checkpoint de arquitectura antigua incompatible (CNN → MLP)
**Síntoma:** `RuntimeError: Unexpected key(s) in state_dict: 'features.conv.0.weight'`  
**Causa:** Checkpoint de arquitectura CNN; `load()` no filtraba claves desconocidas.  
**Solución:** Filtrar tanto mismatches de tamaño como claves desconocidas en `agent.load()`.

---

### Error 9: Valores BWAPI UnitCommandType incorrectos — unidades estáticas
**Síntoma:** Agente envía ATTACK_MOVE pero todas las unidades permanecen inmóviles.  
**Causa:** `CMD_ATTACK_MOVE = 13` (incorrecto → `UnitCommandType::Stop`), `CMD_MOVE = 6` (→ `Research`), etc.  
**Solución:** Corregir todos los valores en `constants.py`.


In [ ]:
# Error 9 fix: valores correctos de BWAPI UnitCommandType (version 4.x)

# INCORRECTO (fallback original con valores erroneos):
WRONG_VALUES = {
    'CMD_MOVE':        6,   # -> BWAPI::Research (!)
    'CMD_ATTACK_MOVE': 13,  # -> BWAPI::Stop    (!)
    'CMD_GATHER':      7,   # -> BWAPI::Upgrade (!)
    'CMD_BUILD':       5,   # -> BWAPI::Morph   (!)
    'CMD_TRAIN':       4,   # correcto
}

# CORRECTO (sc1_rl/torchcraft/constants.py):
CMD_ATTACK_MOVE      = 1   # BWAPI::UnitCommandType::Attack_Move
CMD_BUILD            = 2   # BWAPI::UnitCommandType::Build
CMD_TRAIN            = 4   # BWAPI::UnitCommandType::Train
CMD_MOVE             = 10  # BWAPI::UnitCommandType::Move
CMD_GATHER           = 15  # BWAPI::UnitCommandType::Gather
CMD_RIGHT_CLICK_POS  = 30  # BWAPI::UnitCommandType::Right_Click_Position
CMD_RIGHT_CLICK_UNIT = 31  # BWAPI::UnitCommandType::Right_Click_Unit

print("Valores correctos BWAPI UnitCommandType:")
for name, val in [('Attack_Move',1),('Build',2),('Train',4),('Move',10),
                  ('Gather',15),('Right_Click_Pos',30)]:
    wrong = WRONG_VALUES.get(f'CMD_{name.replace("_","_").upper()}', '?')
    print(f"  {name:25s} = {val:3d}")


### Error 10: Doble ciclo send/recv por paso del entorno
**Síntoma:** Bucle funciona pero consume 2 frames por cada paso del agente.  
**Causa:** `send()` hace el ciclo REQ→REP completo. Llamar `recv()` después envía un NOOP y consume otro frame.  
**Solución:** Eliminar la llamada redundante a `recv()` — solo usar `send(commands)`.

### Error 11: `code=1` = QUIT en TorchCraft (no Attack_Move)
**Síntoma:** Partida termina exactamente a los 3 pasos, siempre, con `msg_type=6` (EndGame).  
**Causa:** `CMD_ATTACK_MOVE=1` usado como `Command.code` (nivel TorchCraft) → `code=1` = QUIT.  
**Solución:** Usar `TC_CMD_UNIT_PROTECTED=21` como `code` y `CMD_ATTACK_MOVE=1` dentro de `args[1]`.


In [ ]:
# Error 10 fix: eliminacion de recv() redundante en step()

# ANTES (incorrecto - 2 frames por paso):
def step_WRONG(self, commands):
    self.tc.send(commands)  # frame N -> N+1
    ok = self.tc.recv()     # frame N+1 -> N+2  <- DOBLE CONSUMO
    state = self.tc.state   # estado del frame N+2, no N+1

# DESPUES (correcto - 1 frame por paso):
def step_CORRECT(self, commands):
    ok = self.tc.send(commands)  # envia y recibe en un ciclo REQ->REP
    state = self.tc.state        # estado del frame N+1

# Error 11 fix: formato correcto del comando

# ANTES (incorrecto - envia QUIT al recibir code=1):
def make_cmd_WRONG(uid, x, y):
    CMD_ATTACK_MOVE = 1
    return [CMD_ATTACK_MOVE, uid, -1, x, y, 0]  # code=1 = QUIT!

# DESPUES (correcto):
TC_CMD_UNIT_PROTECTED = 21
CMD_ATTACK_MOVE_BWAPI = 1

def make_cmd_CORRECT(uid, x, y):
    return [TC_CMD_UNIT_PROTECTED, uid, CMD_ATTACK_MOVE_BWAPI, -1, x, y, 0]
#           ^--- TorchCraft code      ^--- BWAPI UnitCommandType dentro de args


### Error 12: BWEnv reporta `player_id=0` para todas las unidades
**Síntoma:** `units_by_player_id={0: 16}` — ambos equipos bajo el mismo jugador.  
**Causa:** BWEnv en modo micro-scenario consolida el estado de ambos jugadores bajo player_id=0.  
**Solución:** Split posicional por diagonal `x+y` (ver código arriba).

---

### Error 13: Tipo de unidad BWAPI = 101, no 65 (Zealot) ni 66 (Dragoon)
**Síntoma:** Filtro por tipo excluye todas las unidades (`ARMY_TYPES` contiene 65, 66 pero estado reporta 101).  
**Causa:** Discrepancia de versión entre TorchCraft/BWAPI; tipo 101 no documentado.  
**Solución:** Eliminar filtro por tipo para combate; solo excluir edificios, workers y recursos.

---

### Error 14: Socket ZMQ REQ atascado en estado EFSM
**Síntoma:** Todos los pasos fallan con `Operation cannot be accomplished in current state`. Episodios de 1 paso a miles por segundo.  
**Causa:** ZMQ REQ requiere alternancia estricta send→recv. Si recv() falla, el socket queda en "awaiting reply" (EFSM). El socket roto no puede ser reutilizado.

---

### Error 15: Episodios terminan antes de `game_ended` — estado contaminado entre partidas
**Síntoma:** A partir del segundo episodio, split posicional devuelve 13v13 (unidades de la partida anterior mezcladas).  
**Causa:** `combat_over` dispara antes de que BWEnv envíe `EndGame`. `reset()` no llama a `reconnect()` porque `game_ended=False`.

---

### Error 16: Executor comandaba unidades de ambos equipos
**Síntoma:** `ATTACK_MOVE → 15 units` — el agente comanda a ambos equipos.  
**Causa:** Filtro por `player_id == 0` incluye todas las unidades (ver Error 12).  
**Solución:** Añadir split posicional en `CommandExecutor` con `_own_unit_ids` reseteado cada episodio.


In [ ]:
# Error 14 fix: socket ZMQ fresco en reconnect()
# sc1_rl/torchcraft/client.py

# El problema: reutilizar el socket roto no funciona
# Socket REQ en estado EFSM:
#   send() -> ZMQError: Operation cannot be accomplished in current state
#   recv() -> ZMQError: Operation cannot be accomplished in current state
#   (loop infinito de errores)

# La solucion: siempre cerrar el socket viejo y crear uno nuevo
import zmq

class SocketFix:
    def _close_socket(self):
        """Cierra solo el socket ZMQ (linger=0), dejando el contexto vivo."""
        if self._sock is not None:
            try:
                self._sock.close(linger=0)  # linger=0: no esperar mensajes pendientes
            except Exception:
                pass
            self._sock = None

    def reconnect(self):
        self._close_socket()         # <- CLAVE: socket fresco, no reutilizar el roto
        self._sock = self._ctx.socket(zmq.REQ)
        self._sock.setsockopt(zmq.SNDTIMEO, 10_000)
        self._sock.setsockopt(zmq.RCVTIMEO, int(self.connect_timeout * 1000))
        self._sock.setsockopt(zmq.LINGER, 0)
        self._sock.connect(f"tcp://{self.host}:{self.port}")
        # ... handshake normal ...


In [ ]:
# Error 15 fix: drenado de frames + limpieza de unidades stale
# sc1_rl/environment/sc1_env_tc.py

# PARTE 1: dreno de frames en step() cuando combat_over dispara antes de game_ended
def step_drain_fix(self, action):
    # ... (enviar comando, calcular recompensa) ...
    combat_over = (
        n_army  == 0 or
        (n_enemy == 0 and max_enemy > 0)
    )
    if combat_over and not state.game_ended:
        for _ in range(60):  # hasta 60 NOOPs adicionales
            if not self.tc.send([]):  # NOOP
                break
            state = self.tc.state
            if state is None or state.game_ended:
                break
        if state is not None and not state.game_ended:
            state.game_ended = True  # forzar para que reset() llame a reconnect()

# PARTE 2: limpieza de stale units en reset()
def reset_stale_fix(self):
    # Guardar IDs antes de reconectar
    stale_ids = set()
    if self.tc.state is not None:
        stale_ids = {
            u.id for pid_u in self.tc.state.units.values() for u in pid_u.values()
        }

    # ... reconectar, recibir primer frame ...

    # Eliminar stale units del nuevo estado
    if stale_ids and self.tc.state is not None:
        for pid_units in self.tc.state.units.values():
            for uid in list(pid_units.keys()):
                if pid_units[uid].id in stale_ids:
                    del pid_units[uid]

    # Fallback: si hay mas unidades de las esperadas, conservar las N mas recientes (IDs mas altos)
    EXPECTED = 16  # 8v8
    all_units = [
        (u.id, pid, uid)
        for pid, pid_units in self.tc.state.units.items()
        for uid, u in pid_units.items()
    ]
    if len(all_units) > EXPECTED:
        all_units.sort(key=lambda t: t[0], reverse=True)
        keep_ids = {t[0] for t in all_units[:EXPECTED]}
        for pid_units in self.tc.state.units.values():
            for uid in list(pid_units.keys()):
                if pid_units[uid].id not in keep_ids:
                    del pid_units[uid]


## 12. Uso Rápido <a id='uso'></a>

### Requisitos del sistema

- Windows 10 (host), VirtualBox 7.x, Vagrant 2.x
- VM Windows 7 con StarCraft BW 1.16.1 + BWAPI 4.4.0 + BWEnv.dll
- Python 3.13 en el host

### Arranque de la VM y StarCraft

```powershell
# 1. Arrancar la VM
vagrant up

# 2. Dentro de la VM (acceder por RDP o consola VirtualBox):
#    a) Ejecutar ChaosLauncher.exe como Administrador
#    b) Plugin 'BWAPI Injector 4.4.0 [RELEASE]' activo
#    c) Pulsar 'Start' -> StarCraft arranca y navega hasta la partida
#    d) Esperar a que aparezcan las unidades en el mapa
```

### Estructura de archivos

```
Proyecto_Final/
+-- main.py                        # Punto de entrada
+-- config.yaml                    # Hiperparametros y configuracion
+-- zmq_test.py                    # Diagnostico de conexion TorchCraft
+-- requirements.txt               # Dependencias Python
+-- Vagrantfile                    # Configuracion de la VM
|
+-- sc1_rl/
|   +-- torchcraft/
|   |   +-- proto.py               # FlatBuffers TorchCraft puro Python
|   |   +-- client.py              # TorchCraftClient (ZMQ REQ)
|   |   +-- constants.py           # Constantes BWAPI
|   |   +-- action_space.py        # 65 macro-acciones
|   |   +-- command_executor.py    # TCAction -> comandos BWAPI
|   |   +-- state_encoder.py       # GameState -> float32[189]
|   |   +-- reward.py              # TCRewardCalculator
|   +-- environment/
|   |   +-- sc1_env_tc.py          # gymnasium.Env
|   +-- model/
|   |   +-- agent.py               # PPOAgent
|   |   +-- network_mlp.py         # ActorCriticMLP
|   |   +-- trainer.py             # Bucle rollout -> GAE -> PPO update
|   |   +-- memory.py              # RolloutBuffer con GAE
|   +-- logger/
|       +-- action_logger.py       # Logger de acciones y episodios
|
+-- checkpoints/                   # Checkpoints PPO
+-- logs/                          # Logs de entrenamiento
+-- tc_maps/                       # Mapas micro TorchCraft
```


In [ ]:
# Instalacion de dependencias
# Ejecutar en el host Windows (no en Colab)
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install gymnasium numpy pyzmq flatbuffers pyyaml

# En Colab, para instalar las dependencias de Python del proyecto:
required_packages = [
    "torch>=2.1",
    "gymnasium>=0.29",
    "numpy>=1.26",
    "pyzmq>=26",
    "flatbuffers>=24",
    "pyyaml>=6.0",
]
print("Dependencias del proyecto:")
for p in required_packages:
    print(f"  {p}")
print()
print("NOTA: El entrenamiento requiere conexion a BWEnv.dll en la VM (192.168.0.23:11111)")
print("No puede ejecutarse directamente en Google Colab sin la VM.")


In [ ]:
# Comandos de entrenamiento (ejecutar en el host Windows)
training_commands = {
    "nuevo desde cero": "python main.py --fresh",
    "continuar desde checkpoint": "python main.py",
    "checkpoint especifico": "python main.py --resume checkpoints/step_00050000.pt",
    "config alternativa": "python main.py --config config_alt.yaml",
    "verificar conectividad": "python zmq_test.py",
}

print("Comandos disponibles:")
for desc, cmd in training_commands.items():
    print(f"  [{desc}]")
    print(f"    > {cmd}")
    print()

# Salida esperada de zmq_test.py si BWEnv esta corriendo:
expected_output = """
Conectando a TorchCraft en 192.168.0.23:11111...
HandshakeClient enviado - esperando respuesta del servidor (timeout=120 s)...
TorchCraft conectado - map=dragoons_zealots player=0 neutral=11 lag=1
Protocol version: 30
RESPUESTA recibida: XXXX bytes
msg_type=3  (esperado 3=HandshakeServer)
"""
print("Salida esperada de zmq_test.py:")
print(expected_output)


## 13. Recursos y Trabajo Futuro <a id='recursos'></a>

### Software

| Recurso | Versión | Rol |
|---|---|---|
| Python | 3.13 | Runtime principal del agente |
| PyTorch | ≥ 2.1 | Red neuronal y backpropagation |
| Gymnasium | ≥ 0.29 | Interfaz estándar de entorno RL |
| NumPy | ≥ 1.26 | Operaciones matriciales en el buffer de rollout |
| pyzmq | ≥ 26 | Socket ZMQ para comunicación con BWEnv |
| flatbuffers | ≥ 24 | Serialización manual del protocolo TorchCraft |
| VirtualBox | 7.x | Hypervisor para la VM de juego |
| Vagrant | 2.x | Automatización del ciclo de vida de la VM |
| BWAPI | 4.4.0 | Hook en StarCraft BW |
| TorchCraft | v1.4.0 | BWEnv.dll — servidor ZMQ |
| ChaosLauncher | — | Inyector de BWAPI en StarCraft |
| cnc-ddraw | — | Wrapper DirectDraw 256 colores |
| StarCraft BW | 1.16.1 | Entorno de juego |

---

### Trabajo Futuro

- **Convergencia de combate:** señal de recompensa por distancia necesita refinamiento;
  explorar reward shaping más agresivo (*focus fire*, kiting explícito).
- **Centralizar identificación de equipo:** los tres componentes
  (`StateEncoder`, `TCRewardCalculator`, `CommandExecutor`) mantienen su propio
  `_own_unit_ids` independiente. Convendría un objeto `TeamTracker` compartido.
- **Identificación por comportamiento:** en lugar del split posicional, detectar
  el equipo propio observando qué unidades responden a `TC_CMD_UNIT_PROTECTED`.
- **Self-play:** entrenar contra versiones anteriores del agente.
- **Escalado:** combates 10v10, 20v20; diversificar razas.
- **Arquitecturas recurrentes:** LSTM para información parcialmente observable.
- **Estabilidad de reconexión:** reiniciar BWEnv completamente entre episodios
  en lugar de depender de `auto_restart=ON` (origen de los errores 15 y 16).

---

## 14. Referencias

- Schulman, J., et al. (2017). *Proximal Policy Optimization Algorithms*. arXiv:1707.06347.
- Mnih, V., et al. (2015). *Human-level control through deep reinforcement learning*. Nature.
- Synnaeve, G., et al. (2016). *TorchCraft: a Library for Machine Learning Research on RTS Games*. arXiv:1611.00625.
- Vinyals, O., et al. (2019). *Grandmaster level in StarCraft II using multi-agent RL*. Nature.
- BWAPI Development Team. *BWAPI 4.4.0 Documentation*. https://bwapi.github.io
- Flatbuffers Documentation. *Writing a Schema*. https://flatbuffers.dev
